In [2]:
#Load Libraries
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn import metrics
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RepeatedStratifiedKFold
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.feature_selection import RFE
from sklearn.linear_model import Lasso
from sklearn.decomposition import PCA
import plotly.express as px
from collections import Counter
import collections
import math
from tabulate import tabulate

In [3]:
# Read in all dataframes
invitees_df = pd.read_csv(r"C:\Users\mrice\OneDrive\Documents\Data Science - Callum\Projects\Sports Projects\golf_data_engineering_pipeline\masters_players_2025.csv")
masters_results_df = pd.read_csv(r"C:\Users\mrice\OneDrive\Documents\Data Science - Callum\Projects\Sports Projects\golf_data_engineering_pipeline\masters_results.csv")
player_stats_df = pd.read_csv(r"C:\Users\mrice\OneDrive\Documents\Data Science - Callum\Projects\Sports Projects\golf_data_engineering_pipeline\player_stats.csv")
wins_df = pd.read_csv(r"C:\Users\mrice\OneDrive\Documents\Data Science - Callum\Projects\Sports Projects\golf_data_engineering_pipeline\wins.csv")
player_stats_df = player_stats_df.merge(wins_df, how='left', on=['player_name', 'year'])

In [4]:
# Building datasets
full_df = player_stats_df.merge(masters_results_df, how='left', on = ['player_id', 'player_name', 'year'])
full_df = full_df.rename(columns={'position': 'masters_finish'})
# Training data is from 2024 and earlier
train_df = full_df.loc[full_df['year'] <= 2024]
# Prediciting for this year's results is 2024/2025 season
data_2025 = full_df.loc[full_df['year'] >= 2024]
# Prediction dataset is merged with the field of this years competition
prediction_data = invitees_df.merge(data_2025, how='left', on='player_name')
prediction_data.drop(columns=['country', 'masters_finish'], inplace=True)

In [5]:
print(tabulate(train_df, headers='keys', tablefmt='pretty', showindex=False))

+-----------+---------------------------+------+-----------+-------------+-----------+------------+-------+------------+------------+------------+-----------------+----------+----------------+--------------------+--------------------+-------------------+---------------------+--------------------+--------+--------------------+----------------+
| player_id |        player_name        | year | avg_score | bounce_back | drive_acc | drive_dist |  gir  | par3_score | par4_score | par5_score | putts_per_round | scramble | strokes_gained | strokes_gained_arg | strokes_gained_atg | strokes_gained_ot | strokes_gained_putt | strokes_gained_ttg | top_10 | 1st_place_finishes | masters_finish |
+-----------+---------------------------+------+-----------+-------------+-----------+------------+-------+------------+------------+------------+-----------------+----------+----------------+--------------------+--------------------+-------------------+---------------------+--------------------+--------+----

In [6]:
# strip T and reaplce cut with 100
# Use .loc to ensure the operation is on the original DataFrame
train_df.loc[:, 'masters_finish'] = train_df['masters_finish'].str.lstrip("T")
train_df.loc[:, 'masters_finish'] = train_df['masters_finish'].str.replace("CUT", "100")
train_df.loc[:, 'masters_finish'] = train_df['masters_finish'].str.replace("W/D", "100")
train_df.loc[:, 'masters_finish'] = pd.to_numeric(train_df['masters_finish'], errors='coerce')
train_df.loc[:, 'masters_finish'] = train_df['masters_finish'].astype(float)


In [7]:
train_df.loc[:, 'masters_finish'] = train_df['masters_finish'].fillna(100)
train_df.loc[:, 'top_10'] = train_df['top_10'].fillna(0)
train_df.loc[:, '1st_place_finishes'] = train_df['1st_place_finishes'].fillna(0)
print(tabulate(train_df, headers='keys', tablefmt='pretty', showindex=False))


C:\Users\mrice\AppData\Local\Temp\ipykernel_25380\3628564588.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_df.loc[:, 'masters_finish'] = train_df['masters_finish'].fillna(100)


+-----------+---------------------------+------+-----------+-------------+-----------+------------+-------+------------+------------+------------+-----------------+----------+----------------+--------------------+--------------------+-------------------+---------------------+--------------------+--------+--------------------+----------------+
| player_id |        player_name        | year | avg_score | bounce_back | drive_acc | drive_dist |  gir  | par3_score | par4_score | par5_score | putts_per_round | scramble | strokes_gained | strokes_gained_arg | strokes_gained_atg | strokes_gained_ot | strokes_gained_putt | strokes_gained_ttg | top_10 | 1st_place_finishes | masters_finish |
+-----------+---------------------------+------+-----------+-------------+-----------+------------+-------+------------+------------+------------+-----------------+----------+----------------+--------------------+--------------------+-------------------+---------------------+--------------------+--------+----

In [8]:
train_df=train_df.dropna()
print(train_df.shape)

(3798, 22)


In [9]:
def top_10(pos):
    try:
        pos=int(pos)
        if pos <= 10:
            return 1
        else:
            return 0
    except ValueError:
        return 0
    
train_df['top_10_masters'] = train_df['masters_finish'].apply(top_10)

print(train_df[train_df['top_10_masters']==1])

      player_id      player_name  year  avg_score  bounce_back  drive_acc  \
45         1226     Fred Couples  2006     70.789        15.76      54.36   
176        1810   Phil Mickelson  2005     69.386        21.28      58.69   
177        1810   Phil Mickelson  2006     69.500        23.33      58.61   
179        1810   Phil Mickelson  2008     69.167        26.21      55.27   
180        1810   Phil Mickelson  2009     70.218        21.28      52.21   
...         ...              ...   ...        ...          ...        ...   
4357      51634  Sahith Theegala  2023     70.219        24.64      52.85   
4404      52372    Cameron Champ  2022     71.527        19.64      57.79   
4431      52955     Ludvig Åberg  2024     69.988        21.76      62.38   
4510      57366    Cameron Young  2023     70.515        25.11      59.05   
4511      57366    Cameron Young  2024     70.841        23.48      59.61   

      drive_dist    gir  par3_score  par4_score  ...  strokes_gained  \
45 

In [10]:
def winner(x):
    if x == 1:
        win = 1
    else:
        win = 0
    return win

train_df.loc[:, 'masters_win'] = train_df['masters_finish'].apply(winner)
print(tabulate(train_df, headers='keys', tablefmt='pretty', showindex=False))


+-----------+---------------------------+------+-----------+-------------+-----------+------------+-------+------------+------------+------------+-----------------+----------+----------------+--------------------+--------------------+-------------------+---------------------+--------------------+--------+--------------------+----------------+----------------+-------------+
| player_id |        player_name        | year | avg_score | bounce_back | drive_acc | drive_dist |  gir  | par3_score | par4_score | par5_score | putts_per_round | scramble | strokes_gained | strokes_gained_arg | strokes_gained_atg | strokes_gained_ot | strokes_gained_putt | strokes_gained_ttg | top_10 | 1st_place_finishes | masters_finish | top_10_masters | masters_win |
+-----------+---------------------------+------+-----------+-------------+-----------+------------+-------+------------+------------+------------+-----------------+----------+----------------+--------------------+--------------------+--------------

In [11]:
print(train_df[(train_df['player_name'] == 'Phil Mickelson') & (train_df['year'] == 2010)])
train_df.loc[181, 'masters_win'] = 1


     player_id     player_name  year  avg_score  bounce_back  drive_acc  \
181       1810  Phil Mickelson  2010     69.966         19.7      52.66   

     drive_dist    gir  par3_score  par4_score  ...  strokes_gained_arg  \
181       299.1  65.13        3.05        4.01  ...               0.228   

     strokes_gained_atg  strokes_gained_ot  strokes_gained_putt  \
181               0.738              0.185               -0.149   

     strokes_gained_ttg  top_10  1st_place_finishes  masters_finish  \
181               1.151     6.0                 1.0           100.0   

     top_10_masters  masters_win  
181               0            0  

[1 rows x 24 columns]


In [12]:
# Remove rows where 'year' is 2019 or 2016
train_df = train_df[~train_df['year'].isin([2019, 2016])]



In [13]:
columns = list(train_df.columns.values)
predictors = columns[3:]
predictors.remove('top_10_masters')
predictors.remove('masters_win')
predictors.remove('masters_finish')
predictors.remove('top_10')
predictors.remove('1st_place_finishes')
X = train_df[predictors]
y = train_df['masters_win']

over = SMOTE(sampling_strategy=0.1, k_neighbors=3)
under = RandomUnderSampler(sampling_strategy=1)
steps = [('o', over), ('u', under)]
pipeline = Pipeline(steps=steps)
X_train, y_train = pipeline.fit_resample(X, y)

rfc = RandomForestClassifier()
model = rfc.fit(X_train, y_train)

pd.DataFrame(list(zip(X.columns, model.feature_importances_)), columns=['predictor', 'feature_importace'])

,predictor,feature_importace
0,avg_score,0.261638
1,bounce_back,0.023516
2,drive_acc,0.023480
3,drive_dist,0.041726
4,gir,0.019736
5,par3_score,0.019847
6,par4_score,0.045292
7,par5_score,0.020155
8,putts_per_round,0.015912
9,scramble,0.022501


In [14]:
# Random forest no sampling techniques
training = train_df.loc[train_df['year']<=2019]
test = train_df.loc[train_df['year']>2019]
X_train = training[predictors]
y_train = training['top_10_masters']
X_test = test[predictors]
y_test = test['top_10_masters']

rfc = RandomForestClassifier()
model = rfc.fit(X_train, y_train)

y_test_pred = model.predict(X_test)

print('precision:', metrics.precision_score(y_test, y_test_pred))
print('recall:', metrics.recall_score(y_test, y_test_pred))
print('f1:', metrics.f1_score(y_test, y_test_pred))
print('accuarcy:', metrics.accuracy_score(y_test, y_test_pred))

precision: 0.4
recall: 0.15
f1: 0.21818181818181817
accuarcy: 0.955067920585162


In [15]:
# Preicting top 10 with resmapling techniques
training = train_df.loc[train_df['year']<=2019]
testing = train_df.loc[train_df['year']>2019]
X_train = training[predictors]
y_train = training['top_10_masters']
X_test = testing[predictors]
y_test = testing['top_10_masters']

over = SMOTE(sampling_strategy=0.5)
under = RandomUnderSampler(sampling_strategy=0.95)

steps = [('o', over), ('u', under)]
pipeline = Pipeline(steps=steps)

X_train, y_train = pipeline.fit_resample(X_train, y_train)
model = rfc.fit(X_train, y_train)

y_pred = model.predict(X_test)

print('precision:', metrics.precision_score(y_test, y_pred))
print('recall:', metrics.recall_score(y_test, y_pred))
print('f1:', metrics.f1_score(y_test, y_pred))
print('accuarcy:', metrics.accuracy_score(y_test, y_pred))


precision: 0.2736842105263158
recall: 0.65
f1: 0.3851851851851852
accuarcy: 0.9132706374085684


In [16]:
# Best out of box model
training = train_df.loc[train_df['year']<=2019]
testing = train_df.loc[train_df['year']>2019]
X_train = training[predictors]
y_train = training['top_10_masters']
X_test = testing[predictors]
y_test = testing['top_10_masters']

models = {
    'random_forest': RandomForestClassifier(),
    'gradient_boosted': xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss'),
    'SVC': SVC()
}

over = SMOTE(sampling_strategy=0.5)
under = RandomUnderSampler(sampling_strategy=0.95)

results = {}

for name, model in models.items():
    
    steps = [('o', over), ('u', under), ('model', model)]
    pipeline = Pipeline(steps=steps)

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    results[name] = {
        'precision': metrics.precision_score(y_test, y_pred),
        'recall': metrics.recall_score(y_test, y_pred),
        'f1': metrics.f1_score(y_test, y_pred),
        'accuracy': metrics.accuracy_score(y_test, y_pred)
    }
    
for name, scores in results.items():
    print(f"{name} Performance:")
    print(f"Precision: {scores['precision']:.4f}")
    print(f"Recall: {scores['recall']:.4f}")
    print(f"F1 Score: {scores['f1']:.4f}")
    print(f"Accuracy: {scores['accuracy']:.4f}")
    print("-" * 50)



random_forest Performance:
Precision: 0.2500
Recall: 0.6500
F1 Score: 0.3611
Accuracy: 0.9039
--------------------------------------------------
gradient_boosted Performance:
Precision: 0.2584
Recall: 0.5750
F1 Score: 0.3566
Accuracy: 0.9133
--------------------------------------------------
SVC Performance:
Precision: 0.0602
Recall: 0.9750
F1 Score: 0.1134
Accuracy: 0.3626
--------------------------------------------------


In [17]:
# Grid search for best params
training = train_df.loc[train_df['year']<=2019]
testing = train_df.loc[train_df['year']>2019]
X_train = training[predictors]
y_train = training['top_10_masters']
X_test = testing[predictors]
y_test = testing['top_10_masters']

pipeline = Pipeline([
    ('smote', SMOTE()),
    #('under', RandomUnderSampler()),
    ('classifier', SVC())
])

param_grid = {
    'smote__sampling_strategy': [0.2, 0.75, 1.0],
    'smote__k_neighbors': [1, 3, 5],
    #'under__sampling_strategy': [0.3, 0.5, 0.8]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print('Best params:', grid_search.best_params_)

print('precision:', metrics.precision_score(y_test, y_pred))
print('recall:', metrics.recall_score(y_test, y_pred))
print('f1:', metrics.f1_score(y_test, y_pred))
print('accuracy:', metrics.accuracy_score(y_test, y_pred))

Fitting 5 folds for each of 9 candidates, totalling 45 fits
Best params: {'smote__k_neighbors': 1, 'smote__sampling_strategy': 1.0}
precision: 0.05673758865248227
recall: 1.0
f1: 0.10738255033557047
accuracy: 0.3051201671891327


In [18]:
training = train_df.loc[train_df['year']<=2019]
testing = train_df.loc[train_df['year']>2019]
X_train = training[predictors]
y_train = training['top_10_masters']
X_test = testing[predictors]
y_test = testing['top_10_masters']

over = SMOTE(sampling_strategy=0.2, k_neighbors=3)
under = RandomUnderSampler(sampling_strategy=0.7)

steps = [('o', over), ('u', under)]
pipeline = Pipeline(steps=steps)

X_train_res, y_train_res = pipeline.fit_resample(X_train, y_train)

model = SVC()
model.fit(X_train_res, y_train_res)

y_pred = model.predict(X_test)


print('precision:', metrics.precision_score(y_test, y_pred))
print('recall:', metrics.recall_score(y_test, y_pred))
print('f1:', metrics.f1_score(y_test, y_pred))
print('accuarcy:', metrics.accuracy_score(y_test, y_pred))




precision: 0.0
recall: 0.0
f1: 0.0
accuarcy: 0.9582027168234065


C:\Users\mrice\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [19]:
training = train_df.loc[train_df['year']<=2019]
testing = train_df.loc[train_df['year']>2019]
X_train = training[predictors]
y_train = training['masters_win']
X_test = testing[predictors]
y_test = testing['masters_win']

scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)

over = SMOTE(sampling_strategy=0.5, k_neighbors=5)
under = RandomUnderSampler(sampling_strategy=1)

steps = [('o', over), ('u', under)]
pipeline = Pipeline(steps=steps)

X_train_res, y_train_res = pipeline.fit_resample(X_train_std, y_train)

model = SVC(kernel='linear')
model.fit(X_train_res, y_train_res)

y_pred = model.predict(X_test_std)


print('precision:', metrics.precision_score(y_test, y_pred))
print('recall:', metrics.recall_score(y_test, y_pred))
print('f1:', metrics.f1_score(y_test, y_pred))
print('accuarcy:', metrics.accuracy_score(y_test, y_pred))

precision: 0.08333333333333333
recall: 0.75
f1: 0.15
accuarcy: 0.9644723092998955


In [20]:
data_2025=data_2025.dropna()

In [21]:
training = train_df
test = data_2025
X_train = training[predictors]
y_train = training['masters_win']
X_test = test[predictors]

# Scale the data (fit on training data, transform both training and testing data)
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)
        
# Apply SMOTE and RandomUnderSampler to the training data only
over = SMOTE(sampling_strategy=0.2, k_neighbors= 3)
under = RandomUnderSampler(sampling_strategy = 0.5)

steps = [('o', over), ('u', under)]
pipeline = Pipeline(steps=steps)

X_train_mod, y_train_mod = pipeline.fit_resample(X_train_std, y_train)
        
# Train the model on the resampled training data
svc = SVC(kernel = 'linear', probability = True)
model = svc.fit(X_train_mod, y_train_mod)

y_pred = model.predict(X_test_std)
y_pred_proba = model.predict_proba(X_test_std)

#Add Predictions to DF
data_2025['masters_prediction'] = y_pred.tolist()
probs = []
for i in range(0, len(y_pred_proba)):
    prob = y_pred_proba[i,1]
    probs.append(prob)
    
data_2025['prob'] = probs

index = probs.index(np.max(probs))
loc = X_test.iloc[index,:]['avg_score']
data_2025.loc[data_2025['avg_score'] == loc]


,player_id,player_name,year,avg_score,bounce_back,drive_acc,drive_dist,gir,par3_score,par4_score,...,strokes_gained_arg,strokes_gained_atg,strokes_gained_ot,strokes_gained_putt,strokes_gained_ttg,top_10,1st_place_finishes,masters_finish,masters_prediction,prob
3408,33448,Justin Thomas,2024,70.2,18.82,55.83,309.1,65.59,3.0,4.0,...,0.466,0.639,0.135,-0.478,1.24,6.0,0.0,CUT,1,0.964042


In [22]:
winners = data_2025.loc[data_2025['masters_prediction']==1]
print(winners)

      player_id        player_name  year  avg_score  bounce_back  drive_acc  \
2646      28237       Rory McIlroy  2024     69.914        20.75      60.29   
3408      33448      Justin Thomas  2024     70.200        18.82      55.83   
3486      34046      Jordan Spieth  2024     70.844        18.45      62.45   
4062      46046  Scottie Scheffler  2024     68.645        31.75      66.90   
4237      48081  Xander Schauffele  2024     69.137        21.80      60.84   

      drive_dist    gir  par3_score  par4_score  ...  strokes_gained_arg  \
2646       320.2  65.70        3.04        3.98  ...               0.248   
3408       309.1  65.59        3.00        4.00  ...               0.466   
3486       306.9  65.87        3.11        4.00  ...              -0.024   
4062       303.8  73.16        2.98        3.88  ...               0.316   
4237       308.5  69.97        2.92        3.92  ...               0.195   

      strokes_gained_atg  strokes_gained_ot  strokes_gained_putt  \


In [24]:
from tqdm import tqdm
from collections import Counter
import collections

dubs = []

for i in tqdm(range(0, 500), desc="Running simulations"):
    # Split Data
    training = train_df
    test = data_2025
    X_train = training[predictors]
    y_train = training['masters_win']
    X_test = test[predictors]

    # Standardize Predictors
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train)
    X_test_std = scaler.fit_transform(X_test)

    # SMOTE and Undersampling
    over = SMOTE(sampling_strategy=0.2, k_neighbors=3)
    under = RandomUnderSampler(sampling_strategy=0.5)

    steps = [('o', over), ('u', under)]
    pipeline = Pipeline(steps=steps)

    X_train_mod, y_train_mod = pipeline.fit_resample(X_train_std, y_train)

    # Train Model
    svc = SVC(kernel='linear', probability=True)
    model = svc.fit(X_train_mod, y_train_mod)

    y_pred = model.predict(X_test_std)
    y_pred_proba = model.predict_proba(X_test_std)

    # Add Predictions to DF
    data_2025['masters_prediction'] = y_pred.tolist()
    data_2025['prob'] = [prob[1] for prob in y_pred_proba]

    winners = data_2025['player_name'].loc[data_2025['masters_prediction'] == 1].to_list()
    dubs.append(winners)

# Format Results
flat_list = [item for sublist in dubs for item in sublist]
results = Counter(flat_list)
sorted_results = collections.OrderedDict(sorted(results.items(), reverse=True, key=lambda t: t[1]))


Running simulations: 100%|██████████| 500/500 [05:32<00:00,  1.50it/s]


In [25]:
print(sorted_results)

OrderedDict([('Rory McIlroy', 500), ('Justin Thomas', 500), ('Scottie Scheffler', 500), ('Xander Schauffele', 500)])


In [ ]:
training = train_df
test = data_2025
X_train = training[predictors]
y_train = training['masters_win']
X_test = test[predictors]

# Scale the data (fit on training data, transform both training and testing data)
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)
        
# Apply SMOTE and RandomUnderSampler to the training data only
over = SMOTE(sampling_strategy=0.2, k_neighbors= 3)
under = RandomUnderSampler(sampling_strategy = 0.5)

steps = [('o', over), ('u', under)]
pipeline = Pipeline(steps=steps)

X_train_mod, y_train_mod = pipeline.fit_resample(X_train_std, y_train)
        
# Train the model on the resampled training data
svc = SVC(kernel = 'linear', probability = True)
model = svc.fit(X_train_mod, y_train_mod)

y_pred = model.predict(X_test_std)
y_pred_proba = model.predict_proba(X_test_std)

#Add Predictions to DF
data_2025['masters_prediction'] = y_pred.tolist()
probs = []
for i in range(0, len(y_pred_proba)):
    prob = y_pred_proba[i,1]
    probs.append(prob)
    
data_2025['prob'] = probs

index = probs.index(np.max(probs))
loc = X_test.iloc[index,:]['avg_score']
data_2025.loc[data_2025['avg_score'] == loc]


C:\Users\mrice\AppData\Local\Temp\ipykernel_22716\635712107.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_2025['masters_prediction'] = y_pred.tolist()
C:\Users\mrice\AppData\Local\Temp\ipykernel_22716\635712107.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_2025['prob'] = probs


,player_id,player_name,year,avg_score,bounce_back,drive_acc,drive_dist,gir,par3_score,par4_score,...,strokes_gained_arg,strokes_gained_atg,strokes_gained_ot,strokes_gained_putt,strokes_gained_ttg,top_10,1st_place_finishes,masters_finish,masters_prediction,prob
3408,33448,Justin Thomas,2024,70.2,18.82,55.83,309.1,65.59,3.0,4.0,...,0.466,0.639,0.135,-0.478,1.24,6.0,0.0,CUT,1,0.934499
